In [ ]:
import os
import re
import csv
from collections import Counter
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset

#WORD FREQUENCY COMPARISON: CONVERSATIONAL CORPUS (LMSYS) VS GENERAL WEB TEXT (C4)
#this is a comparison of the two raw text corpora, independent of any tokenizer, used to
#fill Table II (top-100 words exclusive to each domain)

#Hugging Face login
load_dotenv("key.env")
login(os.getenv("HF_TOKEN"))

In [ ]:
#load both corpora, restricted to English so the comparison is not confounded by language mix

#conversational corpus: LMSYS train+test (generated with create_corpus.ipynb)
train_corpus = Dataset.from_parquet("train_randomsplit.parquet")
test_corpus = Dataset.from_parquet("test_randomsplit.parquet")
train_en = train_corpus.filter(lambda r: r["language"] == "English")
test_en = test_corpus.filter(lambda r: r["language"] == "English")
conversational_texts = list(train_en["clean_conversation"]) + list(test_en["clean_conversation"])

#general web corpus: C4 english validation split. Pointed at only the 8 "en/c4-validation.*"
#shards (~826 MB) via an explicit data_files pattern: without it, load_dataset("allenai/c4",
#"en", ...) downloads the full "en" config, including the 1024 train shards (828 GB), before
#filtering down to the validation split
web_corpus = load_dataset("allenai/c4", data_files={"validation": "en/c4-validation.*.json.gz"}, split="validation")
web_texts = web_corpus["text"]

In [ ]:
#word definition follows Tadanobu's table2_word_frequency.py exactly (scripts_antiguos/tadanobu/):
#split on whitespace, strip leading/trailing punctuation from each token (internal punctuation,
#e.g. in hyphenated words or contractions, is left untouched), lowercase, and drop tokens that
#are purely numeric. Unicode right/left single quotes are normalized to ASCII apostrophes first,
#since LMSYS text mixes both and leaving them distinct would silently split "don't" counts
#across two spellings and drop the contraction out of the top-100 entirely.
import unicodedata

PUNCT = re.compile(r"^[^\w']+|[^\w']+$", re.UNICODE)


def normalize(text):
    text = unicodedata.normalize("NFKC", text)
    return text.replace("’", "'").replace("ʼ", "'")


def word_counts(texts):
    counts = Counter()
    for text in texts:
        if not text:
            continue
        for raw in normalize(text).split():
            w = PUNCT.sub("", raw).lower()
            if w and not w.isdigit():
                counts[w] += 1
    return counts

In [ ]:
#cached to disk after the first full scan, since the corpora are large and the counts are
#reused by later cells (e.g. checking specific words) without rescanning everything again
import pickle

cache_path = "_word_frequency_cache.pkl"
if os.path.exists(cache_path):
    with open(cache_path, "rb") as f:
        conv_counts, web_counts = pickle.load(f)
else:
    conv_counts = word_counts(conversational_texts)
    web_counts = word_counts(web_texts)
    with open(cache_path, "wb") as f:
        pickle.dump((conv_counts, web_counts), f)

conv_total = sum(conv_counts.values())
web_total = sum(web_counts.values())
print(f"total conversational word tokens: {conv_total}")
print(f"total web word tokens: {web_total}")

In [ ]:
#save the full top-100 of each corpus, for transparency/reproducibility
with open("word_frequency_top100_conversational.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "word", "count", "frequency_pct"])
    for rank, (w, c) in enumerate(conv_counts.most_common(100), start=1):
        writer.writerow([rank, w, c, round(100 * c / conv_total, 4)])

with open("word_frequency_top100_web.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "word", "count", "frequency_pct"])
    for rank, (w, c) in enumerate(web_counts.most_common(100), start=1):
        writer.writerow([rank, w, c, round(100 * c / web_total, 4)])

In [ ]:
#for Table II: top-10 conversational words absent from the web top-100, and vice versa
#(searched among each corpus's top 500 words, since the top-100-exclusive words are not
#necessarily within the other corpus's own top 100 by rank)
#
#name_<n> anonymization placeholders (LMSYS) are excluded here, not from the counting itself,
#matching Tadanobu's recommendation: they would otherwise occupy the top ranks as a
#preprocessing artefact rather than a property of dialogue. Their combined share is reported
#separately for a footnote.
NAME_PLACEHOLDER = re.compile(r"^name_\d+$")
name_share_pct = 100 * sum(c for w, c in conv_counts.items() if NAME_PLACEHOLDER.match(w)) / conv_total
print(f"combined name_* share of conversational words: {name_share_pct:.2f}% (excluded from the table below)")

top_conv_100 = set(w for w, _ in conv_counts.most_common(100))
top_web_100 = set(w for w, _ in web_counts.most_common(100))

conv_only = [(w, c) for w, c in conv_counts.most_common(600) if w not in top_web_100 and not NAME_PLACEHOLDER.match(w)][:10]
web_only = [(w, c) for w, c in web_counts.most_common(600) if w not in top_conv_100][:10]

with open("word_frequency_diff.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "conversational_word", "conversational_freq_pct", "web_word", "web_freq_pct"])
    for rank, ((cw, cc), (ww, wc)) in enumerate(zip(conv_only, web_only), start=1):
        writer.writerow([rank, cw, round(100 * cc / conv_total, 4), ww, round(100 * wc / web_total, 4)])

print("Conversational words not in web top-100 (name_* excluded):")
for w, c in conv_only:
    print(f"  {w}\t{100*c/conv_total:.4f}%")
print("\nWeb words not in conversational top-100:")
for w, c in web_only:
    print(f"  {w}\t{100*c/web_total:.4f}%")

In [ ]:
#rank check for the turn-marker claim discussed in Section IV.A.4 / the Discussion:
#"assistant" and "user" ranks among the top conversational words, if present at all
conv_ranked = conv_counts.most_common()
for target in ["assistant", "user", "it's", "i'm"]:
    rank = next((i for i, (w, _) in enumerate(conv_ranked, start=1) if w == target), None)
    print(f"{target!r}: rank {rank}" if rank else f"{target!r}: not found")

In [ ]:
#cross-check against Tadanobu's finding on his 8MB sample: of 9 common contractions, only
#you're/i'm/it's were richer in conversation, while don't/that's/can't/doesn't/i've/we're were
#richer in general web text - checked here at full corpus scale
contractions = ["don't", "that's", "can't", "doesn't", "i've", "we're", "you're", "i'm", "it's"]
print(f"{'word':<10}{'conversational %':<18}{'web %':<12}{'richer in'}")
for w in contractions:
    cp = 100 * conv_counts.get(w, 0) / conv_total
    wp = 100 * web_counts.get(w, 0) / web_total
    richer = "conversation" if cp > wp else "web"
    print(f"{w:<10}{cp:<18.4f}{wp:<12.4f}{richer}")